# Leetcode 175 
## Combine Two Tables

In [0]:
# https://leetcode.com/problems/combine-two-tables
# Completed at 2026/04/19

In [0]:
# Frameworks
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
import pyspark.sql.functions as F
from datetime import datetime

## Part 1: Parsing the raw data into PySpark Dataframes

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime


def parse_raw_to_df(table_str):
    spark = SparkSession.builder.getOrCreate()

    # Step 1 — clean lines
    lines = []

    for line in table_str.splitlines():
        line = line.strip()

        if not line:
            continue

        if line.startswith("+"):
            continue

        lines.append(line)

    # Step 2 — extract schema
    schema_cols = []
    schema_types = []

    i = 0

    # find schema header
    while i < len(lines):
        if "Column Name" in lines[i]:
            i += 1
            break
        i += 1

    # read schema rows
    while i < len(lines):
        parts = [p.strip() for p in lines[i].split("|") if p.strip()]

        if len(parts) == 2:
            schema_cols.append(parts[0])
            schema_types.append(parts[1])
        else:
            break

        i += 1

    # Step 3 — Spark type mapping
    type_map = {
        "int": IntegerType(),
        "integer": IntegerType(),
        "string": StringType(),
        "varchar": StringType(),
        "char": StringType(),
        "date": DateType()
    }

    fields = []

    for col, typ in zip(schema_cols, schema_types):
        spark_type = type_map.get(typ.lower(), StringType())

        fields.append(
            StructField(col, spark_type, True)
        )

    schema = StructType(fields)

    # Step 4 — find data header
    data_start = None

    for idx, line in enumerate(lines):

        parts = [p.strip() for p in line.split("|") if p.strip()]

        if parts == schema_cols:
            data_start = idx + 1
            break

    if data_start is None:
        raise ValueError("Data section not found")

    # Step 5 — parse data rows
    data_rows = []

    for line in lines[data_start:]:

        parts = [p.strip() for p in line.split("|") if p.strip()]

        if len(parts) != len(schema_cols):
            continue

        converted_row = []

        for val, typ in zip(parts, schema_types):

            typ = typ.lower()

            if typ in ["int", "integer"]:
                converted_row.append(int(val))

            elif typ == "date":
                converted_row.append(
                    datetime.strptime(val, "%Y-%m-%d").date()
                )

            else:
                converted_row.append(val)

        data_rows.append(tuple(converted_row))

    # Step 6 — create dataframe
    df = spark.createDataFrame(data_rows, schema)

    return df

In [0]:
# Initial data tables

# PERSON
r_person = """
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| personId    | int     |
| lastName    | varchar |
| firstName   | varchar |
+-------------+---------+
+----------+----------+-----------+
| personId | lastName | firstName |
+----------+----------+-----------+
| 1        | Wang     | Allen     |
| 2        | Alice    | Bob       |
+----------+----------+-----------+
"""

# ADDRESS
r_address = """
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| addressId   | int     |
| personId    | int     |
| city        | varchar |
| state       | varchar |
+-------------+---------+
+-----------+----------+---------------+------------+
| addressId | personId | city          | state      |
+-----------+----------+---------------+------------+
| 1         | 2        | New York City | New York   |
| 2         | 3        | Leetcode      | California |
+-----------+----------+---------------+------------+
"""
person_df = parse_raw_to_df(r_person)
person_df.createOrReplaceTempView("person")

address_df = parse_raw_to_df(r_address)
address_df.createOrReplaceTempView("address")

In [0]:
person_df.printSchema()
address_df.printSchema()

root
 |-- personId: integer (nullable = true)
 |-- lastName: string (nullable = true)
 |-- firstName: string (nullable = true)

root
 |-- addressId: integer (nullable = true)
 |-- personId: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)



## Part 2: Querying and Solution

Write a solution to report the first name, last name, city, and state of each person in the Person table. If the address of a personId is not present in the Address table, report null instead.

Return the result table in any order.

The result format is in the following example.

![image_1776607346443.png](./image_1776607346443.png "image_1776607346443.png")

### SparkSQL

In [0]:
sol_sql = spark.sql("""
                    SELECT p.firstName, p.Lastname, a.city, a.state 
                    FROM person p
                    LEFT JOIN address a
                    ON p.personId = a.personId                
                    """).show()

+---------+--------+-------------+--------+
|firstName|Lastname|         city|   state|
+---------+--------+-------------+--------+
|    Allen|    Wang|         NULL|    NULL|
|      Bob|   Alice|New York City|New York|
+---------+--------+-------------+--------+



In [0]:
%sql
SELECT p.firstName, p.Lastname, a.city, a.state 
FROM person p
LEFT JOIN address a
ON p.personId = a.personId

firstName,Lastname,city,state
Allen,Wang,null,null
Bob,Alice,New York City,New York


### PySpark


In [0]:
# Pyspark code
combined_df = person_df.join(
    address_df,
    person_df.personId == address_df.personId,
    how = "left"
)

combined_df.select("firstName", "lastName", "city", "state").orderBy("lastName", reverse=False).show()

+---------+--------+-------------+--------+
|firstName|lastName|         city|   state|
+---------+--------+-------------+--------+
|      Bob|   Alice|New York City|New York|
|    Allen|    Wang|         NULL|    NULL|
+---------+--------+-------------+--------+

